# King County House Sales — Correlation & Relationship Analysis
**Dataset:** `kc_house_data.csv`  
Target variable: `price`

In [ ]:
# Dataset: kc_house_data.csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

TARGET = "price"   # ← change if needed
DATA_FILE = "kc_house_data.csv"


## 1. Data Loading and Inspection

In [ ]:
df_raw = pd.read_csv(DATA_FILE)
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns\n")
print("Data types:")
print(df_raw.dtypes)
print("\nFirst 5 rows:")
display(df_raw.head())
print("\nSummary statistics:")
display(df_raw.describe())


## 2. Data Cleaning

In [ ]:
df = df_raw.copy()

# Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Removed {before - len(df)} duplicate rows.")

# Drop non-informative columns
drop_cols = [c for c in ["id", "date"] if c in df.columns]
df.drop(columns=drop_cols, inplace=True)
print(f"Dropped columns: {drop_cols}")

# Convert all remaining columns to numeric where possible
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Before-cleaning correlation snapshot (for Section 11)
corr_before = df.corr(numeric_only=True)

# Impute missing values with median
missing = df.isnull().sum()
print(f"\nMissing values before imputation:\n{missing[missing > 0]}")
for col in df.columns:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)
print("Imputation complete — all NaNs filled with column median.")

# Final numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
feature_cols = [c for c in numeric_cols if c != TARGET]
print(f"\n{len(feature_cols)} feature columns, target = '{TARGET}'")


## 3. Correlation Matrix

In [ ]:
corr_pearson = df[numeric_cols].corr(method="pearson")
corr_spearman = df[numeric_cols].corr(method="spearman")

print("Pearson correlation matrix (shape):", corr_pearson.shape)
display(corr_pearson.style.background_gradient(cmap="coolwarm", axis=None).format("{:.2f}"))


## 4. Heatmap Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
mask = np.zeros_like(corr_pearson, dtype=bool)
mask[np.triu_indices_from(mask)] = True          # upper triangle

sns.heatmap(
    corr_pearson,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.4,
    linecolor="white",
    ax=ax,
    annot_kws={"size": 8},
    cbar_kws={"shrink": 0.8},
)
ax.set_title("Correlation Heatmap (Pearson) — KC House Data", fontsize=14, pad=14)
plt.tight_layout()
plt.savefig("correlation_heatmap.png", bbox_inches="tight")
plt.show()


## 5. Target Correlation Analysis

In [ ]:
target_corr = corr_pearson[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)

print("Correlations with target (price):\n")
print(target_corr.to_string())

fig, ax = plt.subplots(figsize=(9, 7))
colors = ["#d73027" if v > 0 else "#4575b4" for v in target_corr]
ax.barh(target_corr.index[::-1], target_corr.values[::-1], color=colors[::-1], edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Pearson r with price")
ax.set_title("Feature Correlations with Target (price)")
plt.tight_layout()
plt.savefig("target_correlations.png", bbox_inches="tight")
plt.show()

print("\nTop positively correlated:")
print(target_corr[target_corr > 0].head(5))
print("\nTop negatively correlated:")
print(target_corr[target_corr < 0].head(5))


## 6. Pairwise Relationship Plots

In [ ]:
top5 = target_corr.abs().nlargest(5).index.tolist()
print("Top-5 features by |correlation| with price:", top5)

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, feat in zip(axes, top5):
    ax.scatter(df[feat], df[TARGET], alpha=0.15, s=8, color="#2166ac")
    m, b = np.polyfit(df[feat], df[TARGET], 1)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 200)
    ax.plot(x_line, m * x_line + b, color="#d73027", linewidth=1.5)
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel("price" if feat == top5[0] else "", fontsize=9)
    r = corr_pearson.loc[feat, TARGET]
    ax.set_title(f"r = {r:.2f}", fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.suptitle("Top-5 Features vs. Price", y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig("scatter_top5.png", bbox_inches="tight")
plt.show()


In [ ]:
pair_cols = [TARGET] + top5[:4]
g = sns.pairplot(df[pair_cols].sample(2000, random_state=42),
                 diag_kind="kde",
                 plot_kws={"alpha": 0.25, "s": 10},
                 height=2.2)
g.figure.suptitle("Pairplot — Target + Top-4 Features (sample n=2000)", y=1.01, fontsize=11)
plt.savefig("pairplot.png", bbox_inches="tight")
plt.show()


## 7. Strong / Weak Correlation Summary

In [ ]:
STRONG_POS  =  0.70
STRONG_NEG  = -0.70
MODERATE_POS = 0.40
MODERATE_NEG = -0.40

pairs = []
cols = corr_pearson.columns.tolist()
for i, c1 in enumerate(cols):
    for c2 in cols[i+1:]:
        r = corr_pearson.loc[c1, c2]
        pairs.append((c1, c2, r))

strong_pos  = [(a, b, r) for a, b, r in pairs if r >= STRONG_POS]
strong_neg  = [(a, b, r) for a, b, r in pairs if r <= STRONG_NEG]
weak        = [(a, b, r) for a, b, r in pairs if abs(r) < 0.4]

print(f"Strong positive pairs (r ≥ {STRONG_POS}): {len(strong_pos)}")
for a, b, r in sorted(strong_pos, key=lambda x: -x[2]):
    print(f"  {a:20s} ↔ {b:20s}  r = {r:.3f}")

print(f"\nStrong negative pairs (r ≤ {STRONG_NEG}): {len(strong_neg)}")
for a, b, r in sorted(strong_neg, key=lambda x: x[2]):
    print(f"  {a:20s} ↔ {b:20s}  r = {r:.3f}")

print(f"\nWeak pairs (|r| < 0.4): {len(weak)}")


## 8. Top Feature Ranking

In [ ]:
top_features = target_corr.abs().nlargest(5)
ranking = pd.DataFrame({
    "Feature": top_features.index,
    "Abs Correlation": top_features.values,
    "Correlation": target_corr[top_features.index].values,
    "Direction": ["Positive" if v > 0 else "Negative" for v in target_corr[top_features.index].values],
}).reset_index(drop=True)
ranking.index = ranking.index + 1
display(ranking.style
        .format({"Abs Correlation": "{:.3f}", "Correlation": "{:.3f}"})
        .bar(subset=["Abs Correlation"], color="#5fba7d")
        .set_caption("Top-5 Features Most Correlated with Price"))


## 9. Multicollinearity Warning

In [ ]:
MULTI_THRESHOLD = 0.80
feature_pairs = [(a, b, r) for a, b, r in pairs
                 if a != TARGET and b != TARGET and abs(r) >= MULTI_THRESHOLD]

if feature_pairs:
    print("⚠️  Potential multicollinearity detected (|r| ≥ 0.80):\n")
    for a, b, r in sorted(feature_pairs, key=lambda x: -abs(x[2])):
        print(f"  • {a} & {b}: r ≈ {r:.3f}")
        print(f"    → These features carry overlapping information; using both in")
        print(f"      linear regression may inflate coefficient variance.\n")
else:
    print("No severe multicollinearity detected at the 0.80 threshold.")


## 10. Insights

### Key Findings

1. **Strongest predictors of price**  
   `sqft_living` and `grade` show the highest positive correlation with price, meaning larger, higher-grade homes command significantly higher prices. The *grade* variable (reflecting construction quality and design) is almost as predictive as raw living area.

2. **Weakly correlated features**  
   Columns such as `condition`, `yr_built`, and `zipcode` show relatively weak linear relationships with price. While they may still carry non-linear signal, they are unlikely to help much in a simple linear model on their own.

3. **Multicollinearity concerns**  
   `sqft_living` is highly correlated with `sqft_above` and `sqft_living15` (the living area of the nearest 15 neighbours). Including all three in a regression simultaneously risks multicollinearity and inflated standard errors. Prefer one sqft metric or apply regularisation (Ridge/Lasso).

4. **Spearman vs. Pearson**  
   Because `price` is right-skewed, Spearman rank correlations may reveal monotonic relationships that Pearson misses — the rankings are broadly similar here, confirming robust associations.


## 11. Before / After Cleaning Comparison

In [ ]:
target_before = corr_before[TARGET].drop(TARGET, errors="ignore").sort_values(key=abs, ascending=False)
target_after  = corr_pearson[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)

comparison = pd.DataFrame({
    "Before cleaning": target_before,
    "After cleaning":  target_after,
}).dropna()
comparison["Δ"] = (comparison["After cleaning"] - comparison["Before cleaning"]).round(4)

print("Correlation-with-price: before vs. after cleaning")
display(comparison.style.format("{:.3f}").bar(subset=["Δ"], align="mid", color=["#d65f5f", "#5fba7d"]))
